# Demo 11 — Finestra e hop: due effetti distinti

Al primo avvio usare **Run → Run All Cells** (▶▶): la preparazione carica `audiolab`. Poi modificare la cella della demo e rieseguirla con **Shift + Invio**. Ogni riquadro è indipendente.

Le funzioni `al.…` sono quelle della libreria del corso. La prima esecuzione richiede il download del runtime e delle dipendenze.

In [ ]:
# Preparazione automatica: la stessa libreria audiolab dei laboratori.
import sys
from pathlib import Path
ASSET_BASE = None
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["numpy", "scipy", "soundfile", "matplotlib", "plotly==6.3.0", "nbformat>=5.10"])
    from pyodide.http import pyfetch
    from js import location
    import io, zipfile
    ROOT = Path("/home/pyodide/infomm-f0eecc423f2de74c")
    if not (ROOT / "lab-2026/audiolab/__init__.py").exists():
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        ASSET_BASE = str(location.origin) + prefisso + "/_jupyterlite/files"
        url = ASSET_BASE + "/audiolab-course.zip?v=f0eecc423f2de74c"
        risposta = await pyfetch(url)
        if not risposta.ok:
            raise RuntimeError("Impossibile caricare audiolab: " + str(risposta.status))
        with zipfile.ZipFile(io.BytesIO(await risposta.bytes())) as archivio:
            archivio.extractall(ROOT)
    if ASSET_BASE is None:
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        ASSET_BASE = str(location.origin) + prefisso + "/_jupyterlite/files"
else:
    ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "lab-2026/audiolab").is_dir()), None)
    if ROOT is None:
        raise FileNotFoundError("Avviare Jupyter dalla cartella InfoMM-local del corso.")
sys.path.insert(0, str(ROOT / "lab-2026"))
import audiolab as al
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import signal
from IPython.display import Audio, Markdown, display
import time
pio.renderers.default = "plotly_mimetype"
AUDIO = al.AUDIO_DIR
DEMO_ASSETS = ROOT / "lab-2026/interattivo/assets"
SR = FS = 22050
N_FFT, HOP = 2048, 512
BLU, ARANCIO, VERDE = "#1766A5", "#D86822", "#277668"
print("audiolab", al.__version__, "— ambiente pronto")


In [ ]:
y, sr = al.load_audio(AUDIO / "C-major-scale.wav", sr=SR)
configurazioni = [(512, 256, "finestra corta"), (4096, 256, "finestra lunga"),
                  (2048, 128, "hop breve"), (2048, 1024, "hop lungo")]
spettri = []
for lunghezza, hop, titolo in configurazioni:
    w = signal.get_window("hann", lunghezza, fftbins=True)
    S = np.abs(al.analysis.stft(y, n_fft=4096, win_length=lunghezza,
                           hop_length=hop, window=w, center=True)) / w.sum()
    S[1:-1] *= 2
    spettri.append(S)
riferimento = max(S.max() for S in spettri)
fig = make_subplots(rows=2, cols=2,
                    subplot_titles=[f"{t0}: L={L0}, H={H0} | {1000 * L0 / sr:.1f} ms osservati; passo {1000 * H0 / sr:.1f} ms"
                                    for L0, H0, t0 in configurazioni])
for k, (S, (lunghezza, hop, titolo)) in enumerate(zip(spettri, configurazioni)):
    db = al.analysis.amplitude_to_db(S, ref=riferimento, top_db=None)
    r, c = divmod(k, 2)
    f = np.arange(S.shape[0]) * sr / 4096
    tempi = np.arange(S.shape[1]) * hop / sr
    fig.add_trace(go.Heatmap(x=tempi, y=f, z=db, colorscale="Magma", zmin=-80, zmax=0,
                             showscale=(k == 3), colorbar=dict(title="dB")), row=r + 1, col=c + 1)
fig.update_yaxes(range=[0, 3000])
fig.update_layout(height=640, title="Finestra e hop: due effetti distinti")
fig.show()
